# Reranking with a Cross-Encoder

Retrieve broadly, then let a cross-encoder pick the best few.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, hits, top_n=3):
    scores = reranker.predict([(query, h["text"]) for h in hits])
    ranked = sorted(zip(scores, hits), key=lambda p: -p[0])[:top_n]
    return [{**h, "score": float(s)} for s, h in ranked]

query = "Why are cross-encoders only applied to a few candidates?"
candidates = store.search(query, 15)
print("--- before rerank (bi-encoder)"); show(candidates[:3], width=100)
final = rerank(query, candidates)
print("--- after rerank (cross-encoder)"); show(final, width=100)

In [ ]:
print(answer(query, [h["text"] for h in final]))

### LLM reranking (no extra model)
Ask the LLM to score each candidate 0-10. Slower, but works with any model.

In [ ]:
def llm_rerank(query, hits, top_n=3):
    scored = []
    for h in hits:
        out = llm(f"Rate from 0 to 10 how well the passage answers the question. Reply with the number only.\n\nQuestion: {query}\n\nPassage: {h['text']}")
        m = re.search(r"\d+", out)
        scored.append((int(m.group()) if m else 0, h))
    return [h for _, h in sorted(scored, key=lambda p: -p[0])[:top_n]]

show(llm_rerank(query, candidates[:6]), width=100)